# Anhedonic AI — 7B vs 72B Comparison
Side-by-side comparison of reward neuron localization across model scales.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from matplotlib.patches import Patch

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

K_THRESHOLDS = [27, 30, 35, 40, 45, 50, 54]

# Paths
BASE_7B  = '/mnt/mahdipou/models/Anhedonic-AI/Experiment1'
BASE_72B = '/mnt/mahdipou/models/Anhedonic-AI/72-Exp1'

SUBJECTS = [
    'abstract_algebra','anatomy','astronomy','business_ethics',
    'clinical_knowledge','college_biology','college_computer_science',
    'college_mathematics','college_medicine','computer_security',
    'conceptual_physics','econometrics','electrical_engineering',
    'elementary_mathematics','formal_logic','global_facts',
    'high_school_biology','high_school_chemistry','high_school_computer_science',
    'high_school_european_history','high_school_geography',
    'high_school_government_and_politics','high_school_macroeconomics',
    'high_school_mathematics','high_school_microeconomics','high_school_physics',
    'high_school_psychology','high_school_statistics','high_school_us_history',
    'high_school_world_history','human_aging','human_sexuality',
    'international_law','jurisprudence','logical_fallacies','machine_learning',
    'management','marketing','medical_genetics','miscellaneous','moral_disputes',
    'moral_scenarios','nutrition','philosophy','prehistory',
    'professional_accounting','professional_law','professional_medicine',
    'professional_psychology','public_relations','security_studies','sociology',
    'virology','world_religions'
]

print('Config ready.')

In [ ]:
def load_set(path):
    df = pd.read_csv(path)
    return set(zip(df['layer'].tolist(), df['neuron'].tolist()))

def layer_dist(neuron_set, num_layers):
    c = defaultdict(int)
    for (l,_) in neuron_set: c[l] += 1
    return [c.get(l,0) for l in range(num_layers)]

def layer_dist_pct(neuron_set, num_layers):
    d = layer_dist(neuron_set, num_layers)
    t = len(neuron_set) if neuron_set else 1
    return [v/t*100 for v in d]

def band_count(s, l_start, l_end):
    return sum(1 for (l,_) in s if l_start <= l <= l_end)

# ── 7B ────────────────────────────────────────────────────────────────────────
B7 = {
    'orig_reward':  load_set(f'{BASE_7B}/phase4/extraction/universal_reward_neurons.csv'),
    'orig_money':   load_set(f'{BASE_7B}/phase4/extraction/universal_money_neurons.csv'),
    'orig_core':    load_set(f'{BASE_7B}/phase4/extraction/master_incentive_core.csv'),
    'asdiv_reward': load_set(f'{BASE_7B}/evaluation/ASDiv/extraction/asdiv_universal_reward_neurons.csv'),
    'asdiv_money':  load_set(f'{BASE_7B}/evaluation/ASDiv/extraction/asdiv_universal_money_neurons.csv'),
    'asdiv_core':   load_set(f'{BASE_7B}/evaluation/ASDiv/extraction/asdiv_master_incentive_core.csv'),
}
for k in K_THRESHOLDS:
    B7[f'mmlu_k{k}'] = load_set(f'{BASE_7B}/evaluation/mmlu/neurons/mmlu/cross_subject/k{k}.csv')
B7['shared'] = B7['mmlu_k54'] & B7['asdiv_core']
B7['triple'] = B7['mmlu_k54'] & B7['asdiv_core'] & B7['orig_core']

# ── 72B ───────────────────────────────────────────────────────────────────────
B72 = {
    'orig_reward':  load_set(f'{BASE_72B}/neurons/orig/universal_reward_neurons.csv'),
    'orig_money':   load_set(f'{BASE_72B}/neurons/orig/universal_money_neurons.csv'),
    'orig_core':    load_set(f'{BASE_72B}/neurons/orig/master_incentive_core.csv'),
    'asdiv_reward': load_set(f'{BASE_72B}/ASDiv/neurons/asdiv_universal_reward_neurons.csv'),
    'asdiv_money':  load_set(f'{BASE_72B}/ASDiv/neurons/asdiv_universal_money_neurons.csv'),
    'asdiv_core':   load_set(f'{BASE_72B}/ASDiv/neurons/asdiv_master_incentive_core.csv'),
}
for k in K_THRESHOLDS:
    B72[f'mmlu_k{k}'] = load_set(f'{BASE_72B}/MMLU/neurons/cross_subject/k{k}.csv')
B72['shared'] = B72['mmlu_k54'] & B72['asdiv_core']
B72['triple'] = B72['mmlu_k54'] & B72['asdiv_core'] & B72['orig_core']

print('All sets loaded.')
print(f'  7B  orig_core={len(B7["orig_core"])}, asdiv_core={len(B7["asdiv_core"])}, shared={len(B7["shared"])}, triple={len(B7["triple"])}')
print(f'  72B orig_core={len(B72["orig_core"])}, asdiv_core={len(B72["asdiv_core"])}, shared={len(B72["shared"])}, triple={len(B72["triple"])}')

## 1 — Architecture comparison

In [ ]:
arch = pd.DataFrame({
    'Property':       ['Layers','MLP intermediate dim','Parameters','Quantization'],
    '7B':             ['28','3,632','~7B','bfloat16'],
    '72B':            ['80','29,568','~72B','4-bit NF4'],
})
print(arch.to_string(index=False))

# Proportional reward band
print('\nProportional reward band (29-50% network depth):')
print(f'  7B:  layers 8-14  of 28  (L13 = peak)')
print(f'  72B: layers 23-40 of 80  (L38 = peak)')

## 2 — Set sizes: 7B vs 72B

In [ ]:
set_keys = ['orig_core','asdiv_core','mmlu_k27','mmlu_k40','mmlu_k54','shared','triple']
labels   = ['Orig\ncore','ASDiv\ncore','MMLU\nK=27','MMLU\nK=40','MMLU\nK=54','Shared\n(K54∩ASDiv)','Triple\n(K54∩ASDiv∩Orig)']

x  = np.arange(len(set_keys))
w  = 0.35
s7 = [len(B7[k])  for k in set_keys]
s72= [len(B72[k]) for k in set_keys]

fig, ax = plt.subplots(figsize=(14,5))
ax.bar(x-w/2, s7,  w, label='7B',  color='#4878CF', edgecolor='white')
ax.bar(x+w/2, s72, w, label='72B', color='#D65F5F', edgecolor='white')
for xi,(v7,v72) in enumerate(zip(s7,s72)):
    ax.text(xi-w/2, v7*1.05,  f'{v7:,}',  ha='center', fontsize=8)
    ax.text(xi+w/2, v72*1.05, f'{v72:,}', ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Number of neurons')
ax.set_title('7B vs 72B — Neuron set sizes')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.savefig('comparison_set_sizes.png', bbox_inches='tight')
plt.show()

## 3 — Layer distributions: proportional depth comparison

In [ ]:
# Normalize layer index to 0-1 (proportional depth)
def to_proportional(neuron_set, num_layers, n_bins=50):
    depths = [l/(num_layers-1) for (l,_) in neuron_set]
    counts, edges = np.histogram(depths, bins=n_bins, range=(0,1))
    centers = (edges[:-1]+edges[1:])/2
    return centers, counts/len(neuron_set)*100 if neuron_set else counts

fig, axes = plt.subplots(2,3,figsize=(18,10))
pairs = [
    ('Orig core',  'orig_core',  '#4878CF'),
    ('ASDiv core', 'asdiv_core', '#6ACC65'),
    ('MMLU K=27',  'mmlu_k27',   '#D65F5F'),
    ('MMLU K=54',  'mmlu_k54',   '#AA2222'),
    ('Shared',     'shared',     '#B47CC7'),
    ('Triple',     'triple',     '#C4AD66'),
]

for ax, (label, key, color) in zip(axes.flatten(), pairs):
    c7,  v7  = to_proportional(B7[key],  28)
    c72, v72 = to_proportional(B72[key], 80)
    ax.plot(c7,  v7,  color='#4878CF', linewidth=2, label=f'7B  (n={len(B7[key]):,})')
    ax.plot(c72, v72, color='#D65F5F', linewidth=2, label=f'72B (n={len(B72[key]):,})')
    ax.axvspan(0.29, 0.50, alpha=0.08, color='green', label='29-50% depth')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Proportional depth (0=input, 1=output)')
    ax.set_ylabel('% of neurons')
    ax.legend(fontsize=8)

fig.suptitle('7B vs 72B — Neuron depth distributions (proportional)\nGreen band = 29-50% depth', fontsize=12)
plt.tight_layout()
plt.savefig('comparison_proportional_depth.png', bbox_inches='tight')
plt.show()

## 4 — K-threshold neuron counts: 7B vs 72B

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))

# Absolute counts
k_sizes_7b  = [len(B7[f'mmlu_k{k}'])  for k in K_THRESHOLDS]
k_sizes_72b = [len(B72[f'mmlu_k{k}']) for k in K_THRESHOLDS]

axes[0].plot(K_THRESHOLDS, k_sizes_7b,  'o-', color='#4878CF', linewidth=2, label='7B')
axes[0].plot(K_THRESHOLDS, k_sizes_72b, 's-', color='#D65F5F', linewidth=2, label='72B')
axes[0].set_xlabel('K threshold')
axes[0].set_ylabel('Number of neurons')
axes[0].set_title('MMLU K-threshold core sizes')
axes[0].legend()
axes[0].set_yscale('log')
for k,v7,v72 in zip(K_THRESHOLDS, k_sizes_7b, k_sizes_72b):
    axes[0].annotate(f'{v7}', (k,v7), textcoords='offset points', xytext=(-15,5), fontsize=7, color='#4878CF')
    axes[0].annotate(f'{v72}', (k,v72), textcoords='offset points', xytext=(5,5), fontsize=7, color='#D65F5F')

# % overlap with ASDiv core
asdiv_recall_7b  = [len(B7[f'mmlu_k{k}']  & B7['asdiv_core'])  / len(B7['asdiv_core'])  * 100 for k in K_THRESHOLDS]
asdiv_recall_72b = [len(B72[f'mmlu_k{k}'] & B72['asdiv_core']) / len(B72['asdiv_core']) * 100 for k in K_THRESHOLDS]
mmlu_prec_7b  = [len(B7[f'mmlu_k{k}']  & B7['asdiv_core'])  / len(B7[f'mmlu_k{k}'])  * 100 for k in K_THRESHOLDS]
mmlu_prec_72b = [len(B72[f'mmlu_k{k}'] & B72['asdiv_core']) / len(B72[f'mmlu_k{k}']) * 100 for k in K_THRESHOLDS]

axes[1].plot(K_THRESHOLDS, mmlu_prec_7b,  'o-', color='#4878CF', linewidth=2, label='7B % of MMLU_K in ASDiv')
axes[1].plot(K_THRESHOLDS, mmlu_prec_72b, 's-', color='#D65F5F', linewidth=2, label='72B % of MMLU_K in ASDiv')
axes[1].set_xlabel('K threshold')
axes[1].set_ylabel('% of MMLU_K neurons found in ASDiv')
axes[1].set_title('Precision of MMLU K sets vs ASDiv core')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('comparison_k_thresholds.png', bbox_inches='tight')
plt.show()

## 5 — Side-by-side layer distributions for all sets

In [ ]:
plot_keys = ['orig_core','asdiv_core','mmlu_k27','mmlu_k54','shared','triple']
plot_labels = ['Orig core','ASDiv core','MMLU K=27','MMLU K=54','Shared','Triple']

fig, axes = plt.subplots(len(plot_keys), 2, figsize=(20, 4*len(plot_keys)))

for row, (key, label) in enumerate(zip(plot_keys, plot_labels)):
    # 7B
    d7 = layer_dist(B7[key], 28)
    axes[row,0].bar(range(28), d7, color='#4878CF', edgecolor='white', linewidth=0.3)
    axes[row,0].axvspan(7.5, 14.5, alpha=0.1, color='red')
    axes[row,0].axvline(13, color='red', linestyle='--', linewidth=0.8)
    axes[row,0].set_title(f'7B — {label} (n={len(B7[key]):,})', fontsize=10)
    axes[row,0].set_ylabel('Count')
    axes[row,0].set_xlabel('Layer')

    # 72B
    d72 = layer_dist(B72[key], 80)
    axes[row,1].bar(range(80), d72, color='#D65F5F', edgecolor='white', linewidth=0.3)
    axes[row,1].axvspan(22.5, 40.5, alpha=0.1, color='red')
    axes[row,1].axvline(38, color='red', linestyle='--', linewidth=0.8)
    axes[row,1].set_title(f'72B — {label} (n={len(B72[key]):,})', fontsize=10)
    axes[row,1].set_ylabel('Count')
    axes[row,1].set_xlabel('Layer')

fig.suptitle('7B (left) vs 72B (right) — Layer distributions\nRed band = proportional reward zone, dashed = peak layer', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('comparison_layer_distributions.png', bbox_inches='tight')
plt.show()

## 6 — Summary table

In [ ]:
print(f'{'Set':<30} {'7B size':>9}  {'7B L8-14':>9}  {'72B size':>10}  {'72B L23-40':>11}  {'Scale factor':>13}')
print('-'*86)

rows = [
    ('Orig core',     'orig_core'),
    ('ASDiv core',    'asdiv_core'),
    ('MMLU K=27',     'mmlu_k27'),
    ('MMLU K=40',     'mmlu_k40'),
    ('MMLU K=54',     'mmlu_k54'),
    ('Shared (K54∩ASDiv)', 'shared'),
    ('Triple',        'triple'),
]

for label, key in rows:
    s7   = B7[key]
    s72  = B72[key]
    b7   = band_count(s7,  8,  14)
    b72  = band_count(s72, 23, 40)
    sf   = len(s72)/len(s7) if s7 else float('nan')
    print(f'{label:<30} {len(s7):>9,}  {b7:>9,}  {len(s72):>10,}  {b72:>11,}  {sf:>13.2f}x')

print('\nKey insight: reward signal lives at ~29-50% network depth in both models.')
print('Scale factor ~3x for core sets reflects larger MLP dim (29568 vs 3632).')